# Script 1 and 2 - Search Strategies Comparison

#### Objective
This notebook indexes CV data into a qdrant vector database using 3 different search strategies stored in separate in-memory collections:
1. **BM25**
2. **Vector Embedding**: It's a dense model
3. **SPLADE**: It's an sparse model
4. **Hybrid (dense + sparse) with Reciprocal Rank fusion**
5. **Hybrid (dense + sparse) with Distribution-Based Score Fusion (DBSF): DBSF is an evolved version of Relative Score Fusion, which sensitive to outliers.**
6. **Hybrid (dense + bm25) with Reciprocal Rank fusion**
7. **Hybrid (dense + bm25) with Relative Score Fusion**

Search quality across all 7 strategies is evaluated against the ground truth matrix using Hit Rate, MRR@2, MRR@3, Precision, Recall, and NDCG@10.

Note: Since the sparse model and the bm25 algorithm occupy the same search domain (exact or near-exact term matching), combining them is nearly redundant and adds unnecessary complexity.

### 1. Initializing a local client and creating the collections

Initializing Qdrant in  mode for fast testing.

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating the collections for the different search strategies

In [2]:
DENSE_MODEL = {
    "size": 1024,
    "model_name": "microsoft/harrier-oss-v1-0.6b",
}

# # Harrier is too heavy, so testing with a simple one
# DENSE_MODEL = {
#     "size": 384,
#     "model_name": "avsolatorio/GIST-all-MiniLM-L6-v2",
# }
BM25_MODEL = "Qdrant/bm25"
SPARSE_MODEL = "prithivida/Splade_PP_en_v1"
COLLECTION_NAME = "search_strategies"

client.create_collection(
    collection_name=COLLECTION_NAME,
    # Defining semantic search strategies
    vectors_config={
        "dense": models.VectorParams(
            size=DENSE_MODEL["size"],
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        # BM25 and lexical search strategies use IDF (Inverse Document Frequency). 
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF),
        # Neural sparse embedding models (e.g. SPLADE) do not need IDF
        "sparse": models.SparseVectorParams()
    }
)

True

### 2. Loading CV Data and embedding

In [3]:
import os
import json

# Load CV data
with open("../data/cv_extracted_info_eng.json", "r") as file:
    cv_data = json.load(file)

Getting dense vector (Vector embedding)

In [4]:
from sentence_transformers import SentenceTransformer, SparseEncoder
from dotenv import load_dotenv

load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

dense_model = SentenceTransformer(DENSE_MODEL["model_name"], token=HUGGING_FACE_API_KEY)
sparse_model = SparseEncoder("naver/splade-v3")

Loading weights: 100%|██████████| 204/204 [00:00<00:00, 66041.84it/s]


In [5]:
from chonkie import RecursiveChunker

chunker = RecursiveChunker(
    tokenizer=sparse_model.tokenizer, 
    chunk_size=400, # splade is based on BERT, which has 512 tokens max
)

data_points = []

def is_less_than_max_tokens(ref_model, text, applicant_name, chunk_num):
    num_tokens = len(ref_model.tokenizer.encode(text))
    max_tokens = ref_model.max_seq_length
    if num_tokens > max_tokens:
        return False
    return True

def serialize_cv_for_embedding(cv: dict) -> str:
    """Formats CV text specifically for optimal embedding model vectorization."""
    certifications = ", ".join(cv.get("certifications", [])) or "None"
    skills = ", ".join(cv.get("skills", [])) or "None"
    languages = ", ".join(cv.get("languages", [])) or "None"
    
    experience = "\n".join([f"- {exp}" for exp in cv.get("experience", [])])
    education = "\n".join([f"- {edu}" for edu in cv.get("education", [])])

    text = f"""Candidate Name: {cv.get("name")}
Profession: {cv.get("profession")}
Seniority Level: {cv.get("seniority_level")}
Location: {cv.get("location")}
Years of Experience: {cv.get("experience_years")}
Languages: {languages}

About Me:
{cv.get("about_me", "")}

Skills:
{skills}

Certifications:
{certifications}

Work Experience:
{experience}

Education:
{education}
""".strip()
    
    return text

for cv in cv_data:
    cv_text = serialize_cv_for_embedding(cv)
    chunks = chunker(cv_text)
    for c in chunks:
        chunk_str = c.text if hasattr(c, "text") else str(c)
        data_points.append({
            "chunk": chunk_str,
            "cv": cv
        })

chunks = [point["chunk"] for point in data_points]
dense_embeddings = dense_model.encode(chunks).tolist()
sparse_embeddings = sparse_model.encode_document(
    chunks,
    convert_to_sparse_tensor=True # Tensor with no zeroes
).coalesce()



Transforming the sparse embedding into a Qdrant format

In [6]:
doc_indices = sparse_embeddings.indices()[0].tolist()   # Which chunk?
token_indices = sparse_embeddings.indices()[1].tolist() # Which token?
weights = sparse_embeddings.values().tolist()           # The SPLADE score

qdrant_sparse_vectors = [
    {"indices": [], "values": []} for _ in range(len(chunks))
]

for doc_id, token_id, weight in zip(doc_indices, token_indices, weights):
    qdrant_sparse_vectors[doc_id]["indices"].append(token_id)
    qdrant_sparse_vectors[doc_id]["values"].append(weight)

sparse_embeddings = [
    models.SparseVector(indices=vec["indices"], values=vec["values"])
    for vec in qdrant_sparse_vectors
]

### 3. Uploading data points and payload fields to Qdrant Collection

In [7]:
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=idx,
            vector={
                "dense": dense_embeddings[idx],
                "bm25": models.Document(text=data_point["chunk"], model=BM25_MODEL, options={}),
                "sparse": sparse_embeddings[idx]
            
            },
            payload={
                **data_point["cv"],
                "chunk": data_point["chunk"]
            }
        )
        for idx, data_point in enumerate(data_points)
    ]
)   

# Index payload fields. this allows to later search by any
#   of these fields so that multiple chunks belonging to the
#   same candidate CV are grouped together into a single top
#   result per candidate.
fields_to_index = ["location", "seniority_level", "email"]
for field in fields_to_index:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=models.PayloadSchemaType.TEXT
    )


/var/folders/fq/ybvfzjt96mz_ng7wrlmkxpkh0000gn/T/ipykernel_21255/3899118600.py:27: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


### 4. Loading Job Descriptions & Ground Truth Evaluation Matrix

In [8]:
with open("../data/job_descriptions_train_batch.json", "r", encoding="utf-8") as f:
    job_descriptions = json.load(f)

with open("../data/ground_truth_matrix.json", "r", encoding="utf-8") as f:
    ground_truth_matrix = json.load(f)

### 5. Comparing Search Strategies Metrics

In [9]:
import math
import pandas as pd

gt_relevants = {}
for job_id, job in ground_truth_matrix.items():
    relev_cvs = [(eval["name"], eval["score"]) for eval in job["evaluations"] if eval["score"] >= 2]
    gt_relevants[job_id] = len(relev_cvs)

def encode_sparse(qry):
    sparse_embedding = sparse_model.encode_query(qry).coalesce()
    return models.SparseVector(
        indices=sparse_embedding.indices()[0].tolist(),
        values=sparse_embedding.values().tolist()
    )
model_strategies = ["bm25", "dense", "sparse"]
rrf_func = lambda qry: models.FusionQuery(fusion=models.Fusion.RRF)
dbsf_func = lambda qry: models.FusionQuery(fusion=models.Fusion.DBSF)

dense_sparse_prefetch = lambda qry: [model for model in [
            models.Prefetch(
                query=dense_model.encode(qry).tolist(),
                using="dense",
                limit=top_k
            ),
            models.Prefetch(
                query=encode_sparse(qry),
                using="sparse",
                limit=top_k
            )
        ]]

dense_bm25_prefetch = lambda qry: [model for model in [
            models.Prefetch(
                query=dense_model.encode(qry).tolist(),
                using="dense",
                limit=top_k
            ),
            models.Prefetch(
                query=models.Document(text=qry, model=BM25_MODEL),
                using="bm25",
                limit=top_k
            )
        ]]

Q = len(job_descriptions)
hr_thr = 2 # Hit rate relevance threshold
mrr2_thr = 2 # MRR_2 relevance threshold
mrr3_thr = 3 # MRR_3 relevance threshold
pr_thr = 2 # Precision and recall relevance threshold
top_k = 10

metrics = {
    "hit_rate": {},
    "mrr_2": {}, # Mean Reciprocal Rank with a relevance score >= 2
    "mrr_3": {}, # Mean Reciprocal Rank with a relevance score >= 3
    "precision": {},
    "recall": {},
    "ndcg_10": {}, # Normalized Discounted Cumulative Gain at K=10
}


search_strategies = {
    "bm25": {   
        "query_func": lambda qry: models.Document(text=qry, model=BM25_MODEL),
        "prefetch": None
    },
    "dense": {
        "query_func": lambda qry: dense_model.encode(qry).tolist(),
        "prefetch": None
    },
    "sparse": {
        "query_func": encode_sparse,
        "prefetch": None
    },
    "dense_sparse_rrf": {
        "query_func": rrf_func,
        "prefetch": dense_sparse_prefetch
    },
    "dense_sparse_rsf": {
        "query_func": dbsf_func,
        "prefetch": dense_sparse_prefetch
    },
    "dense_bm25_rrf": {
        "query_func": rrf_func,
        "prefetch": dense_bm25_prefetch
    },
    "dense_bm25_rsf": {
        "query_func": dbsf_func,
        "prefetch": dense_bm25_prefetch
    }
}

for strategy_name, config in search_strategies.items():

    hit_rate_accum = 0
    mrr2_accum = 0
    mrr3_accum = 0
    precision_accum = 0
    recall_accum = 0
    ndcg_accum = 0.0

    print("\n" + "="*80)
    print(f" Strategy Name: {strategy_name}")
    print("="*80)

    for job in job_descriptions:
        
        job_desc = job["description"]
        prefetch = config["prefetch"](job_desc) if config["prefetch"] is not None and callable(config["prefetch"]) else None

        # Cleaning up any qdrant failed searches on previous iterations
        client._model_embedder._batch_accumulator.clear()
        result = client.query_points_groups(
            collection_name=COLLECTION_NAME,
            prefetch=prefetch,
            query=config["query_func"](job_desc),
            using=strategy_name if strategy_name in model_strategies else None,
            limit=top_k,
            group_by="email",
            group_size=1
        )

        has_hit = False
        mrr2_rr = 0.0
        mrr3_rr = 0.0
        dcg = 0.0
        relevants = 0

        for rank, group in enumerate(result.groups, start=1):
            eval_matches = [e for e in ground_truth_matrix[job["id"]]["evaluations"] if e["email"] == group.id]
            if eval_matches:
                benchmark_score = eval_matches[0]["score"]
                
                # Calculate DCG for current result
                dcg += benchmark_score / math.log2(rank + 1)
                
                if benchmark_score >= hr_thr:
                    has_hit = True
                
                if benchmark_score >= mrr2_thr and mrr2_rr == 0.0:
                    mrr2_rr = 1.0 / rank
                    
                if benchmark_score >= mrr3_thr and mrr3_rr == 0.0:
                    mrr3_rr = 1.0 / rank

                if benchmark_score >= pr_thr:
                    relevants += 1

        # Ideal DCG (IDCG) for current job from ground truth
        all_eval_scores = sorted(
            [e["score"] for e in ground_truth_matrix[job["id"]]["evaluations"]],
            reverse=True
        )[:top_k]
        idcg = sum(score / math.log2(rank + 1) for rank, score in enumerate(all_eval_scores, start=1))
        
        ndcg_query = (dcg / idcg) if idcg > 0 else 0.0
        ndcg_accum += ndcg_query

        if has_hit:
            hit_rate_accum += 1
            
        mrr2_accum += mrr2_rr
        mrr3_accum += mrr3_rr
        precision_accum += (relevants / top_k) * 100
        recall_accum += (relevants / gt_relevants[job["id"]]) * 100

    metrics["hit_rate"][strategy_name] = hit_rate_accum / Q
    metrics["mrr_2"][strategy_name] = mrr2_accum / Q
    metrics["mrr_3"][strategy_name] = mrr3_accum / Q
    metrics["precision"][strategy_name] = precision_accum / Q
    metrics["recall"][strategy_name] = recall_accum / Q
    metrics["ndcg_10"][strategy_name] = ndcg_accum / Q

df_metrics = pd.DataFrame(metrics)
df_metrics


 Strategy Name: bm25

 Strategy Name: dense

 Strategy Name: sparse

 Strategy Name: dense_sparse_rrf

 Strategy Name: dense_sparse_rsf

 Strategy Name: dense_bm25_rrf

 Strategy Name: dense_bm25_rsf


,hit_rate,mrr_2,mrr_3,precision,recall,ndcg_10
bm25,1.0,0.691667,0.366667,38.0,62.222222,0.683006
dense,1.0,0.840000,0.533333,42.0,60.158730,0.787627
sparse,1.0,0.900000,0.566667,42.0,60.793651,0.778932
dense_sparse_rrf,1.0,0.866667,0.666667,48.0,68.095238,0.832871
dense_sparse_rsf,1.0,0.866667,0.666667,42.0,60.793651,0.798202
dense_bm25_rrf,1.0,0.800000,0.550000,42.0,59.523810,0.778408
dense_bm25_rsf,1.0,0.766667,0.550000,40.0,57.301587,0.764556
